In [1]:
import requests

URL = "https://mosmap.ru/api/api_analitic.php"
API_KEY = "b089de13-5470-4717-9d5f-425f2b4b41a8"

In [58]:
def api_call(lat, lon, radius):
    params = {
        'apikey': API_KEY,
        'longitude': lon,
        'latitude': lat,
        'radius': radius
    }
    
    response = requests.get(URL, params=params)
    data = response.json()
    return data


data = api_call(55.741036, 37.654256, 300)
data

{'latitude': 55.741036,
 'longitude': 37.654256,
 'district_name': 'Таганский',
 'price': {'district_price': '433700',
  'district_price_room1': '400000',
  'district_price_room2': '428400',
  'district_price_room3': '443800',
  'district_price_room4': '503000'},
 'orgs': {'2': {'group_name': 'Медицина', 'count': 17},
  '3': {'group_name': 'Стоматологии', 'count': 5},
  '4': {'group_name': 'Аптеки, оптики', 'count': 23},
  '5': {'group_name': 'Салоны красоты', 'count': 69},
  '6': {'group_name': 'Бани, сауны', 'count': 0},
  '7': {'group_name': 'Фитнес-центры, тренажерные', 'count': 3},
  '8': {'group_name': 'Супермаркеты', 'count': 4},
  '9': {'group_name': 'Гипермаркеты', 'count': 0},
  '10': {'group_name': 'Продуктовые магазины', 'count': 17},
  '11': {'group_name': 'Алкомаркеты, магазины пива', 'count': 9},
  '13': {'group_name': 'Школы, лицеи, гимназии', 'count': 3},
  '14': {'group_name': 'Детские сады', 'count': 0},
  '15': {'group_name': 'Колледжи, институты', 'count': 1},
  '1

In [56]:
import re
import numpy as np
from functools import lru_cache


needed_keys = [
    'latitude', 'longitude',
    'district_name', # название района
    'price', #
    'orgs', # количество организаций
    'zone', # Информация по жилой зоне вокруг точки
    'bcenters', # Расстояние до ближайших бизнес-центров (в метрах) (нужно агрегировать)
    'metro_exits', # Выходы метро и координаты до него
    'traffic1', 'traffic2', 'traffic3', 'traffic4' # 4 типа трафика
]

@lru_cache(maxsize=2048)
def get_data(lat, lon, radius):
    data = api_call(lat, lon, radius)

    # Организации: количество вокруг
    orgs = [d for d in data["orgs"].values()]
    orgs_dict = {f"{d['group_name']}_count_{radius}m": d['count'] for d in orgs}

    # Зона: информация о недвижимости вокруг
    zone_info = {d['name']: d['value'] for d in data["zone"]}
    n_buildings = zone_info['Строений']
    n_living_buildings = zone_info['Жилых домов']
    n_flats = zone_info['Квартир']
    avg_building_age = int(zone_info['Средний возраст домов'].replace(" лет", ""))
    secondary_flat_price = float(zone_info['Стоимость метра жилья вторичка'].replace(" руб.", ""))

    # Бизнес-центры -- расстояния
    bc_distances = [d['distance'] for d in data["bcenters"]]
    min_bc_distance = np.min(bc_distances)
    mean_bc_distance = np.mean(bc_distances)

    # Станции метро: число входов и кол-во станций
    metros = [d for d in data["metro_exits"].values()]
    n_metro_exits = len(metros)
    n_metro_stations = len(set([d["name"] for d in metros]))

    pd_dict = {
        'district_name': data['district_name'],
        
        f'n_buildings_{radius}m': n_buildings,
        f'n_living_buildings_{radius}m': n_living_buildings,
        f'n_flats_{radius}m': n_flats,
        f'avg_building_age_{radius}m': avg_building_age,
        f'secondary_flat_price_{radius}m': secondary_flat_price,

        f'min_bc_distance_{radius}m': min_bc_distance,
        f'mean_bc_distance_{radius}m': mean_bc_distance,

        f'n_metro_exits_{radius}m': n_metro_exits,
        f'n_metro_stations_{radius}m': n_metro_stations,

        f'traffic1_{radius}m': data['traffic1'],
        f'traffic2_{radius}m': data['traffic2'],
        f'traffic3_{radius}m': data['traffic3'],
        f'traffic4_{radius}m': data['traffic4'],
    }

    pd_dict.update(data["price"]) # Не зависит от радиуса
    pd_dict.update(orgs_dict)
    return pd_dict

In [57]:
import pandas as pd

pd.DataFrame(get_data(55.741036, 37.654256, 500), index=[0])

,district_name,n_buildings_500m,n_living_buildings_500m,n_flats_500m,avg_building_age_500m,secondary_flat_price_500m,min_bc_distance_500m,mean_bc_distance_500m,n_metro_exits_500m,n_metro_stations_500m,...,Ветаптеки и ветклиники_count_500m,Магазины цветов_count_500m,"Прачечные, химчистки_count_500m",Детские игровые залы_count_500m,Религия_count_500m,"Пиццерии, суши, столовые_count_500m","Пекарни, кофейни_count_500m","Кафе, бары, рестораны_count_500m",Социальные_count_500m,Банки_count_500m
0,Таганский,344,57,3677,84,546200.0,39,336.764151,7,2,...,2,15,13,1,4,15,30,103,2,7
